# 📈 ShieldPay — 04: Model Evaluation

Deep-dive evaluation of the final XGBoost model:
- Confusion Matrix
- AUC-ROC & Precision-Recall
- Threshold tuning
- SHAP explainability
- Business impact analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib, json, shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

plt.rcParams['figure.facecolor'] = '#0c1118'
plt.rcParams['axes.facecolor']   = '#131b24'
plt.rcParams['axes.edgecolor']   = '#1e2d3d'
plt.rcParams['text.color']       = '#e8edf2'
plt.rcParams['axes.labelcolor']  = '#e8edf2'
plt.rcParams['xtick.color']      = '#6b7a8d'
plt.rcParams['ytick.color']      = '#6b7a8d'
plt.rcParams['grid.color']       = '#1e2d3d'
print('✅ Libraries loaded')

## 1. Load Model & Test Data

In [ ]:
model  = joblib.load('models/fraud_model.pkl')
scaler = joblib.load('models/scaler.pkl')
X_test, y_test = joblib.load('models/test_set.pkl')

with open('models/model_metadata.json') as f:
    meta = json.load(f)

feature_cols = list(pd.read_json('models/feature_columns.json', typ='series'))

print('Model  :', meta['model_type'])
print('Trained:', meta['trained_on'])
print(f'Test set: {len(X_test):,} samples | {y_test.sum()} fraud cases')

## 2. Predictions at Default Threshold (0.5)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], digits=4))
print(f'AUC-ROC          : {roc_auc_score(y_test, y_prob):.4f}')
print(f'Avg Precision    : {average_precision_score(y_test, y_prob):.4f}')

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
labels = np.array([
    [f'TN\n{tn:,}\n(Legit → Legit)', f'FP\n{fp:,}\n(Legit → Fraud)'],
    [f'FN\n{fn:,}\n(Fraud → Legit)', f'TP\n{tp:,}\n(Fraud → Fraud)']
])
custom_colors = np.array([[0.08, 0.22], [0.65, 0.1]])
im = axes[0].imshow(custom_colors, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, labels[i,j], ha='center', va='center',
                     fontsize=11, color='#e8edf2', fontweight='bold', linespacing=1.6)
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Predicted Legit', 'Predicted Fraud'], fontsize=11)
axes[0].set_yticklabels(['Actual Legit', 'Actual Fraud'], fontsize=11)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13, pad=12)

# Metric bars
metrics_dict = {
    'AUC-ROC'  : roc_auc_score(y_test, y_prob),
    'F1 Score' : f1_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
}
bar_colors = ['#00d4aa','#0099ff','#ffa502','#ff4757']
bars = axes[1].barh(list(metrics_dict.keys()), list(metrics_dict.values()),
                    color=bar_colors, edgecolor='none', height=0.5)
axes[1].set_xlim(0, 1.05)
axes[1].set_title('Key Metrics', fontweight='bold', fontsize=13, pad=12)
for bar, val in zip(bars, metrics_dict.values()):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=12, fontweight='bold', color='#e8edf2')

plt.tight_layout()
plt.savefig('eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eval_confusion_matrix.png')

## 4. Threshold Tuning

For fraud detection, **recall is more important than precision** — missing a fraud (FN) costs more than a false alarm (FP). Find the optimal threshold.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.01)
metrics_by_thresh = []

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    metrics_by_thresh.append({
        'threshold': t,
        'f1'       : f1_score(y_test, yp, zero_division=0),
        'precision': precision_score(y_test, yp, zero_division=0),
        'recall'   : recall_score(y_test, yp, zero_division=0),
    })

thresh_df = pd.DataFrame(metrics_by_thresh)
best_f1_thresh = thresh_df.loc[thresh_df['f1'].idxmax(), 'threshold']
best_recall_thresh = thresh_df[thresh_df['recall'] >= 0.90]['threshold'].min()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresh_df['threshold'], thresh_df['f1'],        color='#00d4aa', linewidth=2, label='F1 Score')
ax.plot(thresh_df['threshold'], thresh_df['precision'], color='#0099ff', linewidth=2, label='Precision')
ax.plot(thresh_df['threshold'], thresh_df['recall'],    color='#ff4757', linewidth=2, label='Recall')
ax.axvline(best_f1_thresh, color='#ffa502', linestyle='--', linewidth=1.5,
           label=f'Best F1 threshold = {best_f1_thresh:.2f}')
ax.axvline(0.5, color='#6b7a8d', linestyle=':', linewidth=1, label='Default threshold = 0.50')
ax.set_title('Metrics vs Decision Threshold', fontweight='bold', fontsize=13)
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.legend(fontsize=10)
ax.set_xlim([0.1, 0.9]); ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('eval_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best F1 threshold      : {best_f1_thresh:.2f}')
row = thresh_df[thresh_df['threshold'] == best_f1_thresh].iloc[0]
print(f'  F1={row.f1:.4f}  Precision={row.precision:.4f}  Recall={row.recall:.4f}')

## 5. SHAP Explainability

**SHAP** (SHapley Additive exPlanations) tells us *why* the model made each prediction — essential for trust and compliance.

In [ ]:
# Use a sample for speed (SHAP on full test set can be slow)
sample_idx = np.random.choice(len(X_test), size=500, replace=False)

if hasattr(X_test, 'values'):
    X_sample = X_test.values[sample_idx]
else:
    X_sample = X_test[sample_idx]

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# For binary XGBoost, shap_values may be list[2] or array
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')
print('Top features by mean |SHAP|:')
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feature_cols)
print(mean_abs.nlargest(10).to_string())

In [ ]:
# SHAP Summary Bar Plot
fig, ax = plt.subplots(figsize=(10, 7))
mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=feature_cols).nlargest(15).sort_values()
bars = ax.barh(mean_shap.index, mean_shap.values, color='#00d4aa', edgecolor='none', alpha=0.85)
ax.set_title('SHAP — Mean |Feature Impact| on Fraud Prediction', fontweight='bold', fontsize=13)
ax.set_xlabel('Mean |SHAP value|')
for bar, val in zip(bars, mean_shap.values):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, color='#e8edf2')
plt.tight_layout()
plt.savefig('eval_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eval_shap_summary.png')

In [ ]:
# SHAP Beeswarm (dot plot) — shows direction of impact
shap.summary_plot(sv, X_sample,
                  feature_names=feature_cols,
                  plot_type='dot',
                  max_display=15,
                  show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontweight='bold', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('eval_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eval_shap_beeswarm.png')

## 6. Single Prediction Explanation

In [ ]:
# Explain a single fraudulent transaction
fraud_indices = np.where(y_test.values == 1)[0]
idx = fraud_indices[0]   # first fraud in test set

single = X_test.values[[idx]] if hasattr(X_test, 'values') else X_test[[idx]]
sv_single = explainer.shap_values(single)
if isinstance(sv_single, list): sv_single = sv_single[1]

prob = model.predict_proba(single)[0][1]
print(f'Transaction index : {idx}')
print(f'True label        : FRAUD')
print(f'Predicted prob    : {prob:.4f}  →  {"FRAUD ⚠" if prob>0.5 else "Legit ✓"}')
print()
contrib = pd.Series(sv_single[0], index=feature_cols).sort_values(key=abs, ascending=False)
print('Top 10 SHAP contributions:')
print(contrib.head(10).to_string())

shap.waterfall_plot(
    shap.Explanation(
        values=sv_single[0],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
        feature_names=feature_cols
    ), show=False
)
plt.title('SHAP Waterfall — Single Fraud Prediction', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('eval_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eval_shap_waterfall.png')

## 7. Business Impact Summary

In [ ]:
# Assume average fraud loss €500 per undetected fraud
avg_fraud_loss = 500
total_fraud    = int(y_test.sum())
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

money_saved  = tp * avg_fraud_loss
money_missed = fn * avg_fraud_loss
false_alarms = fp

print('=' * 50)
print('💰 BUSINESS IMPACT ANALYSIS')
print('=' * 50)
print(f'Total fraud cases in test  : {total_fraud}')
print(f'Fraud cases CAUGHT (TP)    : {tp}   → €{money_saved:,} saved')
print(f'Fraud cases MISSED (FN)    : {fn}   → €{money_missed:,} lost')
print(f'False alarms (FP)          : {fp}   → {false_alarms} wasted reviews')
print()
print(f'Fraud detection rate       : {tp/total_fraud*100:.1f}%')
print(f'Money recovery rate        : {money_saved/(money_saved+money_missed)*100:.1f}%')
print()

# Final scorecard
with open('models/model_metadata.json') as f: meta = json.load(f)
print('📋 FINAL MODEL SCORECARD')
print('=' * 50)
for k, v in meta.items():
    if k != 'model_type':
        print(f'  {k:<20}: {v}')
print()
print('🎉 Evaluation complete! Your model is ready for the FastAPI backend.')
print('   → Copy models/ folder next to main.py and run: uvicorn main:app --reload')